# 01 · Coleta

Baixa as internações do SIH/SUS (arquivo RD) do RN para os 12 meses de 2024, direto do FTP do DATASUS, e os nomes dos municípios pela API do IBGE.

**Saídas**
- `data/processed/rd_rn_2024.parquet`: uma linha por internação, só as colunas usadas no projeto
- `data/processed/municipios_rn.csv`: código IBGE de 6 dígitos e nome

**Dependências:** `pandas`, `pyarrow`, `pyreaddbc`, `dbfread`

**Nota:** cada arquivo RD corresponde a um mês de *competência* (quando a AIH foi processada), não ao mês da internação. Um arquivo pode conter internações de meses anteriores.

In [1]:
from ftplib import FTP
from pathlib import Path

import pandas as pd
from dbfread import DBF
from pyreaddbc import dbc2dbf

UF = "RN"
UF_IBGE = 24
ANO = 2024
MESES = range(1, 13)

FTP_HOST = "ftp.datasus.gov.br"
FTP_DIR = "/dissemin/publicos/SIHSUS/200801_/Dados"

COLUNAS = [
    "ANO_CMPT", "MES_CMPT",    # competência
    "MUNIC_RES", "MUNIC_MOV",  # origem e destino do fluxo
    "CNES",                    # hospital
    "DIAG_PRINC",              # CID-10
    "CAR_INT",                 # caráter da internação (eletiva, urgência...)
    "DIAS_PERM", "VAL_TOT",
]

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Download

In [2]:
def nome_arquivo(mes: int) -> str:
    return f"RD{UF}{ANO % 100:02d}{mes:02d}.dbc"

with FTP(FTP_HOST) as ftp:
    ftp.login()
    ftp.cwd(FTP_DIR)
    for mes in MESES:
        destino = RAW / nome_arquivo(mes)
        if destino.exists():
            continue
        parcial = destino.with_suffix(".part")
        with open(parcial, "wb") as f:
            ftp.retrbinary(f"RETR {destino.name}", f.write)
        parcial.rename(destino)
        print("baixado:", destino.name)

baixado: RDRN2401.dbc
baixado: RDRN2402.dbc
baixado: RDRN2403.dbc
baixado: RDRN2404.dbc
baixado: RDRN2405.dbc
baixado: RDRN2406.dbc
baixado: RDRN2407.dbc
baixado: RDRN2408.dbc
baixado: RDRN2409.dbc
baixado: RDRN2410.dbc
baixado: RDRN2411.dbc
baixado: RDRN2412.dbc


## 2. Leitura

`.dbc` é um `.dbf` comprimido. O `pyreaddbc` descomprime para `.dbf` e o `dbfread` lê os registros.

In [3]:
def ler_rd(dbc: Path) -> pd.DataFrame:
    dbf = dbc.with_suffix(".dbf")
    if not dbf.exists():
        dbc2dbf(str(dbc), str(dbf))
    registros = DBF(dbf, encoding="iso-8859-1", load=False)
    return pd.DataFrame({c: r[c] for c in COLUNAS} for r in registros)

df = pd.concat([ler_rd(RAW / nome_arquivo(m)) for m in MESES], ignore_index=True)
df.shape

(216884, 9)

## 3. Tipos

In [4]:
for col in ["MUNIC_RES", "MUNIC_MOV", "CNES", "DIAG_PRINC", "CAR_INT"]:
    df[col] = df[col].astype(str).str.strip()

df["ANO_CMPT"] = df["ANO_CMPT"].astype(int)
df["MES_CMPT"] = df["MES_CMPT"].astype(int)

## 4. Verificação

In [5]:
assert (df["ANO_CMPT"] == ANO).all()
assert sorted(df["MES_CMPT"].unique()) == list(MESES)

codigo_valido = r"\d{6}"
print("MUNIC_RES inválidos:", (~df["MUNIC_RES"].str.fullmatch(codigo_valido)).sum())
print("MUNIC_MOV inválidos:", (~df["MUNIC_MOV"].str.fullmatch(codigo_valido)).sum())

df.groupby("MES_CMPT").size()

MUNIC_RES inválidos: 0
MUNIC_MOV inválidos: 0


MES_CMPT
1     17493
2     17164
3     17878
4     17727
5     17920
6     18716
7     19293
8     19768
9     18797
10    19005
11    17887
12    15236
dtype: int64

## 5. Municípios do RN (IBGE)

A API do IBGE usa códigos de 7 dígitos; o DATASUS usa os 6 primeiros.

In [11]:
url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_IBGE}/municipios"
municipios = pd.read_json(url)[["id", "nome"]]
municipios["cod6"] = municipios["id"].astype(str).str[:6]
municipios = municipios[["cod6", "nome"]]

assert len(municipios) == 167
municipios.head()

,cod6,nome
0,240010,Acari
1,240020,Assú
2,240030,Afonso Bezerra
3,240040,Água Nova
4,240050,Alexandria


In [12]:
nomes = municipios.set_index("cod6")["nome"]

df["NOME_RES"] = df["MUNIC_RES"].map(nomes)
df["NOME_MOV"] = df["MUNIC_MOV"].map(nomes)

assert df["NOME_MOV"].notna().all() 
print(f"Residentes fora do RN (sem nome): {df['NOME_RES'].isna().mean():.1%}")

Residentes fora do RN (sem nome): 0.9%


## 6. Salvar

In [13]:
df.to_parquet(PROCESSED / f"rd_{UF.lower()}_{ANO}.parquet", index=False)
municipios.to_csv(PROCESSED / "municipios_rn.csv", index=False)

In [14]:
df = pd.read_parquet(PROCESSED / f"rd_{UF.lower()}_{ANO}.parquet")

In [15]:
df

,ANO_CMPT,MES_CMPT,MUNIC_RES,MUNIC_MOV,CNES,DIAG_PRINC,CAR_INT,DIAS_PERM,VAL_TOT,NOME_RES,NOME_MOV
0,2024,1,240310,240310,2476487,O820,01,4,705.11,Currais Novos,Currais Novos
1,2024,1,240310,240310,2476487,O458,02,2,689.11,Currais Novos,Currais Novos
2,2024,1,241450,240800,2503689,I602,02,3,463.21,Umarizal,Mossoró
3,2024,1,240990,240800,2503689,I64,02,8,527.21,Pendências,Mossoró
4,2024,1,240020,240800,2503689,K800,02,9,308.33,Assú,Mossoró
...,...,...,...,...,...,...,...,...,...,...,...
216879,2024,12,241460,241460,2407973,N179,01,2,246.89,Upanema,Upanema
216880,2024,12,241460,241460,2407973,J189,01,3,582.42,Upanema,Upanema
216881,2024,12,241460,241460,2407973,N188,01,3,449.65,Upanema,Upanema
216882,2024,12,241475,241475,2409259,A09,02,1,332.90,Venha-Ver,Venha-Ver
